# Horse Racing Results Predictor

The American professional gambler [Bill Benter](https://en.wikipedia.org/wiki/Bill_Benter) is said to have made earned nearly $1 billion through the development of one of the most successful analysis computer software programs in the horse racing market.

Bill published his techniques in the paper [Computer-Based Horse Race Handicapping and Wagering Systems](https://www.gwern.net/docs/statistics/decision/1994-benter.pdf).

The [YouTube Video by Ken Jee](https://www.youtube.com/watch?v=KEeUR8UDy-s) outlines how he did it, how difficult it was, and discusses whether it is likely to be able to replicate this feat today (hint: Ken thinks it highly unlikely for a number of reasons).

Inspired by video, this notebook examines the possibility of replicating Bill's success using data from modern day UK races.

**NOTE: This is a fun examination of the technique the can be used in predicting races. It is not intended to be accurate or valid. The author accepts no responsibility for the correctness, completeness or quality of the information provided. Please do not use this information to place any real-world bets. Gambling odds are always skewed in favour of the bookmaker and you will lose in the long run.**

### Step 1: Load and process the historic race data

Data is loaded via `features.loader.load_results` (all `Results_*.csv` files, in chronological order) and enriched via `FeaturePipeline`, which performs all categorical encoding and horse/jockey/trainer statistics calculation. This is the same load/process path used by `build_features`, so the notebook always reflects the production feature definitions — there is no inline feature-engineering logic here.

In [ ]:
import gc
import os

import numpy as np
import pandas as pd

from race_analytics.data_path import get_data_path
from race_analytics.features import FeaturePipeline, load_results
from race_analytics.features.transforms import going_categories

In [ ]:
_DATA_PATH = get_data_path()

results = load_results(_DATA_PATH)
# For faster interactive exploration you can work with a recent slice, e.g.:
# results = results[results["Off"] >= "2025-01-01"].reset_index(drop=True)

pipeline = FeaturePipeline()
races = pipeline.process(results)
print(f"Processed {len(races)} rows across {races.shape[1]} columns")
gc.collect()

### Analyse factors to understand if they have influence on the outcome of races.

Bill Benter suggested the following attributes:

Current condition:

- performance in recent races
- time since last race
- recent workout data
- age of horse

Past performance:

- finishing position in past races
- lengths behind winner in past races
- normalized times of past races

Adjustments to past performance:

- strength of competition in past races
- weight carried in past races
- jockey's contribution to past performances
- compensation for bad luck in past races
- compensation for advantageous or disadvantageous post position in past races

Present race situational factors:

- weight to be carried
- today's jockey's ability
- advantages or disadvantages of the assigned post position

Preferences which could influence the horse's performance in today's race:

- distance preference
- surface preference (turf vs dirt)
- condition of surface preference (wet vs dry)
- specific track preference

In [ ]:
races.columns

In [ ]:
len(races[races["KnownHorseAndJockey"] == True])

### Evaluating previous horse performance

Now that we have calculated stats for each horse based on the previous races, we can try to figure out if any of the attributes we have relating to previous performance correlate to a horses likelihood of beating another horse in the next race.

Possible performance attributes are:

- Racing post rating
- Official rating
- Top speed rating
- Odds
- Average relative position (i.e. position divided by number of runners in race)

In [ ]:
def calculate_correlation_of_race_attribute(attribute: str) -> np.float64:
    head = races[["RaceId", "HorseId", "FinishingPosition", attribute]]
    cross = pd.merge(head, head, on="RaceId")
    head2head = cross[cross["HorseId_x"] > cross["HorseId_y"]].copy().dropna()
    head2head["XBeatsY"] = (
        head2head["FinishingPosition_x"] < head2head["FinishingPosition_y"]
    )
    head2head["Relative"] = head2head[f"{attribute}_x"] - head2head[f"{attribute}_y"]
    return head2head[["XBeatsY", "Relative"]].corr(method="spearman")["XBeatsY"][
        "Relative"
    ]

In [ ]:
# rp_corr = calculate_correlation_of_race_attribute("LastRaceRacingPostRating")
# gc.collect()

# or_corr = calculate_correlation_of_race_attribute("LastRaceOfficialRating")
# gc.collect()

# ts_corr = calculate_correlation_of_race_attribute("LastRaceTopSpeedRating")
# gc.collect()

# odds_corr = calculate_correlation_of_race_attribute("LastRaceDecimalOdds")
# gc.collect()
# lr_corr = calculate_correlation_of_race_attribute("LastRaceAvgRelFinishingPosition")
# gc.collect()

# print(f"Correlation for Racing Post Rating {rp_corr}, Official Rating {or_corr}, Top Speed Rating {ts_corr}, Odds {odds_corr}, Avg finish {lr_corr}")

#### Conclusions

Racing post rating most strongly correlates with next race performance, other rating attributes not so much.
Official rating correlation is particularly poor. The problem with all these rating is that there are many rows where the values are undefined.

As expected odds and average finishing position are OK negative correlation indicators of prior performance (i.e. lower values correlate to finishing better). The benefits of these indicators is that they are always available for all horses.

However, all these attributes are actually relatively weak on their own (finishing position being the best with a correlation of -0.249). The predictive power is likely to be in combination with other factors.

### Evaluating jockey performance

In [ ]:
# wp_corr = calculate_correlation_of_race_attribute("JockeyWinPercentage")
# pp_corr = calculate_correlation_of_race_attribute("JockeyTop3Percentage")
# fp_corr = calculate_correlation_of_race_attribute("JockeyAvgRelFinishingPosition")
# print(f"Correlation for Win percentage {wp_corr}, place percentage {pp_corr}, average finishing position {fp_corr}")

#### Conclusions

Win and place percentage are OK positive correlation indicator of prior performance. Average finishing position is an OK negative correlation indicator of prior performance (i.e. lower values correlate to finishing better).

However, all these attributes are actually relatively weak on there own (finishing position being the best with a correlation of -0.136). The predictive power is likely to be in combination with other factors.

### Examples

In [ ]:
# oriental_lilly_races = races[races["HorseId"] == 1439510]
# oriental_lilly_races[["Off", "HorseId", "HorseName", "NumberOfPriorRaces", "FinishingPosition", "DaysRested", "LastRaceAvgRelFinishingPosition"] + [f"LastRace{going}" for going in going_categories]]

In [ ]:
# 6901 84857
# james_races = races[(races["JockeyId"] == 6901)].dropna()
# james_races[["Off", "JockeyId", "JockeyName", "JockeyNumberOfPriorRaces", "FinishingPosition", "HorseCount", "JockeyAvgRelFinishingPosition", "JockeyWinPercentage", "JockeyTop3Percentage"]]

### Output

In [ ]:
races.info()

In [ ]:
pipeline.save_race_features(os.path.join(_DATA_PATH, "Race_Features.csv"))